# 06. Reduced models (party identity, Figure 2-C/D/E/F, Figure S4)

Same data preparation and training setup as `05_final_model.ipynb` (XGBoost, native missing-value handling,
5-fold CV × 20 repetitions, survey weights), with different feature sets. Variable metadata
(`type`, `attitude`, `category`) come from sheet `XGBVARS` of `inputs/top_40_features_m3.xlsx`
(the 66 variables of the final model).

| Analysis | Feature set | Output prefix | Figure |
|---|---|---|---|
| Top 3 after 2020 | the 3 most important variables of each ballot in 2021–2024 (final-model SHAP) | `4_1_2020s_Parsimon_top3_only` | 2-F |
| Attitudes only | variables with `attitude == 1` | `4_1_Final_Att_Only` | 2-D (and the Attitude line of 2-E) |
| Without attitudes | variables with `attitude == 0` | `4_1_Final_Att_Excluded` | 2-C |
| Single variable | `letin1a` (number of immigrants) | `4_1_Single_letin1a` | S4 |
| Variable categories | non-attitude variables by `category`: Demography / SES / Lifestyle + Social Capital | `4_1_Cat_{Demography,SES,LifestyleSoccap}` | 2-E |

**Input**: `data/gss7224_r3.dta`, `data/occ10-to-egp-class-crosswalk.csv`, `inputs/top_40_features_m3.xlsx`,
`outputs/3_final/3_FINAL_M3_Top40_partyid_1_analysis_all_importance_weighted.parquet` (from `05_final_model.ipynb`)

**Output**: `outputs/4_parsimonious/`

## Data preparation (same as `05_final_model.ipynb`)

In [ ]:
import pandas as pd
df_origin = pd.read_stata("data/gss7224_r3.dta", convert_categoricals=False)

# Merge survey-experiment versions of the same item
variable_groups = [
    # 1972-2018 vars
    # a
    ('amicable', 'amicably', 'amicablz'),
    # b
    ('bible', 'bibley'),
    ('blnumok', 'blnumoky'),
    # c
    ('class', 'classy'),
    ('clean', 'cleany', 'cleanz'),
    ('colaff', 'colaffy'),
    ('conarmy', 'conarmyy'),
    ('conbus', 'conbusy'),
    ('conclerg', 'conclergy'),
    ('coneduc', 'coneducy'),
    ('confed', 'confedy'),
    ('confinan', 'confinay'),
    ('conjudge', 'conjudgy'),
    ('conlabor', 'conlabory'),
    ('conlegis', 'conlegiy'),
    ('conmedic', 'conmediy'),
    ('conpress', 'conpresy'),
    ('consci', 'consciy'),
    ('consider', 'considey', 'considez'),
    ('control', 'controly', 'controlz'),
    ('contv', 'contvy'),
    ('courts', 'courtsy'),
    # d
    ('discaff', 'discaffy'),
    ('divlaw', 'divlawy'),
    # e
    ('eqwlth', 'eqwlthy'),
    # f
    ('fepol', 'fepoly'),
    # g
    ('genecanx', 'genecany'),
    ('getahead', 'getaheay'),
    ('grass', 'grassy'),
    # h
    ('hinumok', 'hinumoky'),
    ('honest', 'honesty', 'honestz'),
    # i
    ('interest', 'interesy', 'interesz'),
    # j
    ('judgment', 'judgmeny', 'judgmenz'),
    # m
    ('manners', 'mannersy', 'mannersz'),
    # n
    ('nataid', 'nataidy', 'nataidz'),
    ('natarms', 'natarmsy', 'natarmsz'),
    ('natcity', 'natcityy', 'natcityz'),
    ('natcrime', 'natcrimy', 'natcrimz'),
    ('natdrug', 'natdrugy', 'natdrugz'),
    ('nateduc', 'nateducy', 'nateducz'),
    ('natenvir', 'natenviy', 'natenviz'),
    ('natfare', 'natfarey', 'natfarez'),
    ('natheal', 'nathealy', 'nathealz'),
    ('natmass', 'natmassz'),
    ('natpark', 'natparkz'),
    ('natrace', 'natracey', 'natracez'),
    ('natroad', 'natroadz'),
    ('natsoc', 'natsocz'),
    ('natspac', 'natspacy', 'natspacz'),
    ('newsfrom', 'newsfrmy'),
    # o
    ('obeys', 'obeysy', 'obeysz'),
    # p
    ('polviews', 'polviewx', 'polviewy'),
    ('posslq', 'posslqy'),
    ('prayer', 'prayery', 'prayerx'),
    # r
    ('responsi', 'responsy', 'responsz'),
    ('role', 'roley', 'rolez'),
    # s
    ('scifrom', 'scifromy'),
    ('seeksci', 'seeksciy'),
    ('studious', 'studiouy', 'studiouz'),
    ('success', 'successy', 'successz'),
    # t
    ('trust', 'trusty'),
    # u
    ('uswar', 'uswary'),

    # 2024 vars
    # Traditional Wording Experiments
    ('marsame', 'marsamey'),
    ('posslq', 'posslqy'),
    ('racdif1', 'racdif1x'),
    ('racdif2', 'racdif2x'),
    ('racdif3', 'racdif3x'),
    ('racdif4', 'racdif4x'),
    ('racdif5', 'racdif5x'),
    ('xmovie', 'xmoviey'),

    # Volunteered Responses Experiments
    ('bible', 'biblev', 'biblenv'),
    ('courts', 'courtsv', 'courtsnv'),
    ('discaffw', 'discaffwv', 'discaffwnv'),
    ('divlaw', 'divlawv', 'divlawnv'),
    ('fair', 'fairv', 'fairnv'),
    ('fepol', 'fepolv', 'fepolnv'),
    ('fucitzn', 'fucitznv', 'fucitznnv'),
    ('getahead', 'getaheadv', 'getaheadnv'),
    ('grass', 'grassv', 'grassnv'),
    ('helpful', 'helpfulv', 'helpfulnv'),
    ('kidssol', 'kidssolv', 'kidssolnv'),
    ('postlife', 'postlifev', 'postlifenv'),
    ('prayer', 'prayerv', 'prayernv'),
    ('racopen', 'racopenv', 'racopennv'),
    ('reliten', 'relitenv', 'relitennv'),
    ('trust', 'trustv', 'trustnv'),
    ('uscitzn', 'uscitznv', 'uscitznnv'),

    # Gridded Variables Experiments
    ('abany', 'abanyg'),
    ('abdefect', 'abdefectg'),
    ('abhlth', 'abhlthg'),
    ('abnomore', 'abnomoreg'),
    ('abpoor', 'abpoorg'),
    ('abrape', 'abrapeg'),
    ('absingle', 'absingleg'),
    ('suicide1', 'suicide1g'),
    ('suicide2', 'suicide2g'),
    ('suicide4', 'suicide4g'),

    # Gender-neutral Wording Experiments
    ('colcom', 'colcomy'),
    ('famdif16', 'famdif16y'),
    ('letdie1', 'letdie1y'),
    ('libath', 'libathy'),
    ('libcom', 'libcomy'),
    ('libmslm', 'libmslmy'),
    ('librac', 'libracy'),
    ('polabuse', 'polabusey'),
    ('polattak', 'polattaky'),
    ('polhitok', 'polhitoky'),
    ('spkath', 'spkathy'),
    ('spkcom', 'spkcomy'),
    ('spkmslm', 'spkmslmy'),
    ('spkrac', 'spkracy'),

    # found later
    ('libhomo', 'libhomoy'),
    ('spkhomo', 'spkhomoy')
]

# --- Merge each group into one variable ---
df1 = df_origin.copy()
original_cols_to_drop = set() # set avoids duplicates

for group in variable_groups:
    # Keep only the group members present in the data
    existing_vars = [var for var in group if var in df1.columns]

    if not existing_vars:
        continue

    # New column name from the first variable of the group (e.g. nataid_c)
    base_name = existing_vars[0]
    combined_col = f"{base_name.lower()}_c"

    # Combine
    # Start from the first variable and fill its missing values with the other versions
    combined_series = df1[existing_vars[0]]
    for i in range(1, len(existing_vars)):
        combined_series = combined_series.combine_first(df1[existing_vars[i]])

    # Add the merged series to the data
    df1[combined_col] = combined_series

    # Mark the original columns for removal
    original_cols_to_drop.update(existing_vars)

# --- Drop the original (unmerged) variables ---
df1 = df1.drop(columns=list(original_cols_to_drop))


import pandas as pd
import numpy as np



# Target variables
df = df1.copy()
df = df[df['partyid'].notna()]

# partyid
# 0 strong democrat ~ 3 independent ~ 6 strong republican, 7 other party

df = df.assign(
    partyid_1 = np.select(
        [df['partyid'].isin([0, 1, 2]), df['partyid'].isin([4, 5, 6])],
        [1, 2],
        default=0
    ),
    partyid_2 = np.select(
        [df['partyid'].isin([0, 1]), df['partyid'].isin([5, 6])],
        [1, 2],
        default=0
    ),
    partyid_3 = np.select(
        [df['partyid'].isin([0]), df['partyid'].isin([6])],
        [1, 2],
        default=0
    )
)

# --- Convert occ10 and parallel variables to EGP class (based on GSS MR125) ---
print("\n--- EGP class conversion: start ---")
crosswalk_file = 'data/occ10-to-egp-class-crosswalk.csv'
try:
    df_egp_crosswalk = pd.read_csv(crosswalk_file)
    print(f"Loaded '{crosswalk_file}'.")

    # Make crosswalk occ10 numeric
    df_egp_crosswalk['occ10'] = pd.to_numeric(df_egp_crosswalk['occ10'], errors='coerce')
    # Keep only the EGP 10-class code for the merge
    crosswalk_mini = df_egp_crosswalk[['occ10', 'egp10_10']].copy()

    # --- 1. Respondent (R): 10-, 11- and 12-class versions ---
    print("Processing Respondent (R)...")
    df['occ10'] = pd.to_numeric(df['occ10'], errors='coerce')
    df = pd.merge(df, crosswalk_mini, on='occ10', how='left')

    # Classes reassigned to class IV (self-employed): IIIa, IIIb, V, VI, VIIa
    egp_base_classes_for_iv = [3, 4, 8, 9, 10]

    # Respondent's EGP 11-class and 12-class
    if all(col in df.columns for col in ['wrkslf', 'numemps', 'egp10_10']):
        df['wrkslf'] = pd.to_numeric(df['wrkslf'], errors='coerce')
        df['numemps'] = pd.to_numeric(df['numemps'], errors='coerce')

        # 11-Class (egp10_11)
        df['egp10_11'] = df['egp10_10']
        condition_11 = (df['egp10_10'].isin(egp_base_classes_for_iv) & (df['wrkslf'] == 1))
        df.loc[condition_11, 'egp10_11'] = 5 # 5 = class IVab

        # 12-Class (egp10_12)
        df['egp10_12'] = df['egp10_10']
        condition_12a = (condition_11 & (df['numemps'] > 0)) # employers
        condition_12b = (condition_11 & (df['numemps'] == 0)) # own-account workers
        df.loc[condition_12a, 'egp10_12'] = 5 # 5 = class IVa
        df.loc[condition_12b, 'egp10_12'] = 6 # 6 = class IVb
        print("Respondent (R): EGP 10-, 11- and 12-class created.")
    else:
        print("Warning: respondent 'wrkslf' or 'numemps' not found; skipping 11/12-class.")

    # --- 2. Parallel variables (spouse, parents, etc.): up to 11-class ---
    # (occ_code, self_employ_code, new_egp10_col_name, new_egp11_col_name)
    individuals_to_process = {
        'sp': ('spocc10', 'spwrkslf', 'sp_egp10_10', 'sp_egp10_11'),
        'pa': ('paocc10', 'pawrkslf', 'pa_egp10_10', 'pa_egp10_11'),
        'ma': ('maocc10', 'mawrkslf', 'ma_egp10_10', 'ma_egp10_11'),
        'kd': ('kdocc10', 'kdwrkslf', 'kd_egp10_10', 'kd_egp10_11'),
        'sb': ('sbocc10', 'sbwrkslf', 'sb_egp10_10', 'sb_egp10_11'),
        'sphed': ('sphedocc10', 'sphedslf', 'sphed_egp10_10', 'sphed_egp10_11'), # spouse's parent uses 'sphedslf'
    }

    for prefix, (occ_col, slf_col, egp10_col, egp11_col) in individuals_to_process.items():
        if occ_col in df.columns:
            print(f"--- Processing {prefix} ({occ_col}) ---")

            # Harmonize type
            df[occ_col] = pd.to_numeric(df[occ_col], errors='coerce')

            # Step A: merge the 10-class code
            temp_crosswalk = crosswalk_mini.rename(columns={'occ10': occ_col, 'egp10_10': egp10_col})
            df = pd.merge(df, temp_crosswalk, on=occ_col, how='left')

            # Step B: 11-class code (when the self-employment variable 'slf_col' exists)
            if slf_col in df.columns:
                df[slf_col] = pd.to_numeric(df[slf_col], errors='coerce')
                df[egp11_col] = df[egp10_col] # copy the 10-class code

                condition_11_parallel = (
                    df[egp10_col].isin(egp_base_classes_for_iv) &
                    (df[slf_col] == 1)
                )
                df.loc[condition_11_parallel, egp11_col] = 5 # 5 = class IVab
                print(f"Created '{egp10_col}' and '{egp11_col}'.")
            else:
                print(f"Warning: '{slf_col}' not found; creating '{egp10_col}' only.")
        else:
            print(f"Skipping {prefix}: '{occ_col}' not found.")

    print("--- EGP class conversion: done ---")

except FileNotFoundError:
    print(f"Error: {crosswalk_file} not found.")
except KeyError as e:
    print(f"Error: required column not found. {e}")
except Exception as e:
    print(f"Unexpected error: {e}")

# --- End of EGP conversion ---


# Recodes
# --- 1. 'finalter': swap 2 and 3 (2 -> 3, 3 -> 2) ---
if 'finalter' in df.columns:
    # Mixing numeric and string choices in np.select returns strings, so convert to numeric before swapping
    finalter_num = pd.to_numeric(df['finalter'], errors='coerce')
    df['finalter'] = finalter_num.replace({2: 3, 3: 2})
    print("Column 'finalter' processed: 2s and 3s swapped.")
else:
    print("Warning: Column 'finalter' not found. Skipping.")

# --- 2. 'class_c': 5 -> NA ---
if 'class_c' in df.columns:
    df['class_c'] = df['class_c'].replace([5, '5'], np.nan)
    df['class_c'] = pd.to_numeric(df['class_c'], errors='coerce')
    print("Column 'class_c' processed: 5s replaced with NA and converted to numeric.")
else:
    print("Warning: Column 'class_c' not found. Skipping.")


# --- 3. 'old1': 98 -> NA ---
if 'old1' in df.columns:
    df['old1'] = df['old1'].replace([98, '98'], np.nan)
    df['old1'] = pd.to_numeric(df['old1'], errors='coerce')
    print("Column 'old1' processed: 98s replaced with NA and converted to numeric.")
else:
    print("Warning: Column 'old1' not found. Skipping.")


# --- 4. 'kidssol_c': 6 (no children) -> NA (the item only applies to respondents with children) ---
if 'kidssol_c' in df.columns:
    df['kidssol_c'] = df['kidssol_c'].replace([6, '6'], np.nan)
    df['kidssol_c'] = pd.to_numeric(df['kidssol_c'], errors='coerce')
    print("Column 'kidssol_c' processed: 6s (no children) replaced with NA and converted to numeric.")
else:
    print("Warning: Column 'kidssol_c' not found. Skipping.")




# Dictionary of data by year
df_by_year = {year: data for year, data in df.groupby('year')} # {year: data}

for year, df_year in df_by_year.items():
    df_by_year[year] = df_year.dropna(axis=1, how='all') # drop columns that are entirely missing

## Prediction model (same as `05_final_model.ipynb`)

In [ ]:
import os

# Output folder of this notebook
OUT_DIR = "outputs/4_parsimonious"

import pandas as pd
import numpy as np
import shap
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import os

# --- Core function: train/evaluate a model for every (ballot, model, year) ---
def run_single_ml_iteration_weighted_top(
    data_for_analysis_dict, # input data: {(ballot, model): {year: DataFrame}}
    target_col='partyid_1', # target column
    random_seed=123,        # random seed for the model and KFold (set differently in each run)
    run_number=1,           # run index; out-of-fold SHAP values are saved only when it is 1
    use_weights=True,       # apply GSS weights (wtssnrps/wtssps) to training and evaluation
    output_folder_name=None # folder for the out-of-fold SHAP files (None = current folder)
):
    """
    For each (ballot, model) and year, train and evaluate the model with 5-fold CV
    and return the SHAP importance of each fold.

    When run_number == 1, the test-fold SHAP values of all folds are also collected
    and saved as out-of-fold (OOF) SHAP values covering all rows in output_folder_name.
    (This gives SHAP values for the full sample; only run 1 is used.)
    """
    scores_list = []
    importances_list = []
    dimensions_list = []
    feature_usage_list = []

    # Keys are (ballot, model); values are {year: DataFrame}.
    for (ballot, model_name), year_dict in data_for_analysis_dict.items():

        # This notebook only uses XGBoost; any other value means the upstream data are malformed, so stop.
        if model_name != 'XGBoost':
            raise ValueError(f"Unknown model_name: '{model_name}' (ballot={ballot})")

        # Samples and features differ by year, so a separate model is trained for every year.
        for year, df_original in year_dict.items():
            dataset_identifier = f"({ballot}, {model_name}) - {year}"  # identifier for logs/errors

            # OOF SHAP file names (used only in run 1)
            safe_filename_base = f"SHAP_OOF_run1_{ballot}_{model_name}_{year}"

            # Guard against duplicated index values: they could make the .loc-based indexing
            # below select the wrong rows, so reset to 0..n-1.
            df = df_original.copy()
            if not df.index.is_unique: df.reset_index(drop=True, inplace=True)

            # Raise immediately if the data frame is empty or lacks target_col.
            if df.empty:
                raise ValueError(f"{dataset_identifier} - empty data frame.")
            if target_col not in df.columns:
                raise ValueError(f"{dataset_identifier} - target column '{target_col}' not found.")

            # Respondents with a missing or 0 target are dropped.
            df_filtered = df[df[target_col].notna() & (df[target_col] != 0)].copy()
            # No cases after filtering means no valid respondents in this year/ballot;
            # stop instead of skipping silently.
            if df_filtered.empty:
                raise ValueError(f"{dataset_identifier} - no cases left after filtering '{target_col}' != 0.")

            # Re-index to 0..n-1 after filtering
            # (needed below to map KFold positions to row positions within the filtered data for the OOF output)
            df_filtered.reset_index(drop=True, inplace=True)

            y_original = df_filtered[target_col]

            # GSS weights make the sample represent the population in training and evaluation.
            # wtssnrps from 2004 on, wtssps before; equal weights (1) if neither exists.
            weights_original = pd.Series(np.ones(len(df_filtered)), index=df_filtered.index)
            if use_weights:
                if 'wtssnrps' in df_filtered.columns and pd.to_numeric(year, errors='coerce') >= 2004:
                    weights_original = df_filtered['wtssnrps']
                elif 'wtssps' in df_filtered.columns:
                    weights_original = df_filtered['wtssps']
                weights_original = weights_original.fillna(1)

            # 5-fold CV needs at least 20 cases (>= 4 per fold)
            # and both classes. Otherwise raise an error so that
            # problems with the sample (year/subgroup) are noticed.
            if len(df_filtered) < 20 or y_original.nunique() < 2:
                raise ValueError(
                    f"{dataset_identifier} - too few cases ({len(df_filtered)}) or "
                    f"too few classes of '{target_col}' ({y_original.nunique()})."
                )

            # Columns excluded from the features (X):
            # - potential_targets: the target itself (avoid leakage)
            # - essential_cols: weights/metadata (not features)
            potential_targets = [target_col]
            essential_cols = ['wtssnrps', 'wtssps', 'ballot', 'final_weight', 'zodiac']

            all_cols_to_drop = potential_targets + essential_cols
            # Columns differ by year, so only columns present in df_filtered are dropped.
            features_to_drop = [col for col in set(all_cols_to_drop) \
                                if col in df_filtered.columns]

            X = df_filtered.drop(columns=features_to_drop, errors='ignore')
            # No features means the data for this run were built incorrectly, so raise.
            if X.shape[1] == 0:
                raise ValueError(
                    f"{dataset_identifier} - no features. "
                    f"None of the selected features exist in this year/ballot."
                )


            # 'nom' (categorical) features were already converted to category dtype,
            # so the dtype identifies continuous vs. categorical features.
            categorical_cols = X.select_dtypes(include='category').columns.tolist()
            # X only drops columns from df_filtered and keeps its index,
            # so y_original and weights_original can be used as they are.
            y_encoded = LabelEncoder().fit_transform(y_original)
            n_classes = len(np.unique(y_encoded))
            feature_names = X.columns.tolist()
            # --- End of data preparation ---


            # --- Model ---
            # enable_categorical=True: columns with category dtype are handled natively as categorical
            if model_name == 'XGBoost':
                model = xgb.XGBClassifier(
                    eval_metric='logloss',
                    random_state=random_seed,
                    enable_categorical=True
                )
            else:
                raise ValueError(f"Unknown model_name: '{model_name}'")
            # --- End of model ---

            # In run 1, the test-fold SHAP values/data/indices of each fold are collected
            # and saved as out-of-fold (OOF) files after the loop.
            if run_number == 1:
                oof_shap_list = []
                oof_xtest_list = []
                oof_indices_list = []

            # 5-fold CV: collect scores (accuracy/f1/roc_auc) and SHAP importance in every fold.
            kfold = KFold(n_splits=5, shuffle=True, random_state=random_seed)
            fold_scores = {'accuracy': [], 'f1_macro': [], 'roc_auc_ovr': []}
            fold_importances = []

            for fold_i, (train_index_rel, test_index_rel) in enumerate(kfold.split(X, y_encoded)):
                # kfold.split returns positions 0..n-1; they are mapped through X.index
                # to index values (identical to positions here) and used with .loc.
                train_indices_abs = X.index[train_index_rel]
                test_indices_abs = X.index[test_index_rel]

                X_train, X_test = X.loc[train_indices_abs], X.loc[test_indices_abs]
                y_train, y_test = y_encoded[train_index_rel], y_encoded[test_index_rel]
                weights_train, weights_test = weights_original.loc[train_indices_abs], weights_original.loc[test_indices_abs]

                # --- Train ---
                fit_params = {'sample_weight': weights_train} if use_weights else {}
                # A training failure (e.g. y_train with a single class, so base_score cannot be computed)
                # means the fold/year/subgroup sample is too small.
                # Catching it with try/except would aggregate partially filled fold results
                # that do not properly reflect the sample,
                # so the exception is propagated to stop the analysis.
                model.fit(X_train, y_train, **fit_params)


                # --- Predict and score ---
                # Failures in prediction/scoring (e.g. a single-class y_test in roc_auc_score)
                # indicate an invalid analysis, so the exception is propagated.
                y_pred = model.predict(X_test)
                y_proba = model.predict_proba(X_test)

                fold_scores['accuracy'].append(accuracy_score(y_test, y_pred, sample_weight=weights_test))
                fold_scores['f1_macro'].append(f1_score(y_test, y_pred, average='macro', sample_weight=weights_test))

                if n_classes > 2:
                    fold_scores['roc_auc_ovr'].append(roc_auc_score(y_test, y_proba, multi_class='ovr', average='macro', sample_weight=weights_test))
                else:
                    fold_scores['roc_auc_ovr'].append(roc_auc_score(y_test, y_proba[:, 1], average='macro', sample_weight=weights_test))
                # --- End of prediction/scoring ---


                # --- SHAP ---
                # SHAP values measure each feature's contribution to the predictions;
                # the fold averages give the final feature importance.
                # With >= 20 cases and 5 folds, X_test is never empty.
                # If it is, the analysis is broken, so raise instead of filling with NaN.
                if X_test.empty:
                    raise ValueError(f"{dataset_identifier} Fold {fold_i} - X_test is empty.")

                # A SHAP failure indicates a serious problem with the model/input data,
                # so the exception is propagated.
                explainer = shap.TreeExplainer(
                    model,
                    feature_perturbation='tree_path_dependent'
                )

                shap_values_output = explainer.shap_values(
                    X_test,
                    check_additivity=False
                )
                # --- End of SHAP ---


                # --- Harmonize the output format of shap_values() ---
                shap_values_to_process = None

                if isinstance(shap_values_output, list) and len(shap_values_output) == n_classes:
                    shap_values_to_process = shap_values_output[1]
                elif isinstance(shap_values_output, np.ndarray):
                    shap_values_to_process = shap_values_output

                # An unexpected SHAP format or None indicates a problem with the weights/input data,
                # so raise an error to stop the analysis.
                if shap_values_to_process is None:
                    raise ValueError(
                        f"{dataset_identifier} Fold {fold_i} - SHAP values not computed or "
                        f"in an unexpected format. (Type: {type(shap_values_output)})"
                    )

                mean_abs_shap = np.abs(shap_values_to_process).mean(axis=0)
                if len(mean_abs_shap) != len(feature_names):
                    raise ValueError(
                        f"{dataset_identifier} Fold {fold_i} - SHAP dimension mismatch. "
                        f"(SHAP: {len(mean_abs_shap)}, Feat: {len(feature_names)})"
                    )
                fold_importances.append(mean_abs_shap)

                # Keep OOF data (run 1 only)
                if run_number == 1:
                    oof_shap_list.append(shap_values_to_process)
                    oof_xtest_list.append(X_test)
                    oof_indices_list.append(test_indices_abs)
                # --- End of SHAP processing ---

            # --- End of KFold loop ---

            # --- Combine and save OOF data (after the KFold loop, run 1 only) ---
            # Failures when saving/combining OOF SHAP values (array or file IO errors)
            # make the outputs unreliable, so the exception is propagated.
            # If the fold loop finished normally, oof_shap_list is always filled.
            if run_number == 1:
                # 1. Concatenate the fold pieces
                combined_shap = np.concatenate(oof_shap_list, axis=0)
                combined_xtest = pd.concat(oof_xtest_list, axis=0)
                combined_indices = np.concatenate(oof_indices_list, axis=0)

                # 2. Restore the original row order
                sort_order = np.argsort(combined_indices)
                final_oof_shap_values = combined_shap[sort_order]
                final_oof_xtest = combined_xtest.iloc[sort_order]

                # 3. Create the folder and file paths
                output_dir = "." # default: current folder
                if output_folder_name:
                    output_dir = output_folder_name
                    if not os.path.exists(output_dir):
                        os.makedirs(output_dir)
                        print(f"--- Creating OOF folder: {output_dir} ---")

                # Build paths with os.path.join
                shap_filename = os.path.join(output_dir, f"{safe_filename_base}_values.npy")
                xtest_filename = os.path.join(output_dir, f"{safe_filename_base}_xtest.parquet")

                np.save(shap_filename, final_oof_shap_values)
                final_oof_xtest.to_parquet(xtest_filename)

                print(f"  OOF SHAP data saved (run={run_number}):")
                print(f"   - SHAP values: {shap_filename} (shape: {final_oof_shap_values.shape})")
                print(f"   - X_test: {xtest_filename} (shape: {final_oof_xtest.shape})")
            # --- End of OOF saving ---


            # --- Collect results ---
            # Errors when collecting results mean the data frames are malformed even though
            # the steps above finished, so the exception is propagated.
            # 1. Scores
            # (If the fold loop finished normally, each metric has 5 values = n_splits.)
            for metric, values in fold_scores.items():
                scores_list.append({
                    'ballot': ballot,
                    'model': model_name,
                    'year': year,
                    'run': run_number,
                    'metric': metric,
                    'avg_score': np.mean(values),
                    'std_score': np.std(values)
                })

            # 2. Feature importance (SHAP)
            # (If the fold loop finished normally, fold_importances has 5 filled entries = n_splits.)
            importances_array = np.array(fold_importances)
            avg_importances = np.nanmean(importances_array, axis=0)

            # A feature whose SHAP importance is NaN in every fold indicates a SHAP problem;
            # indicates a problem, so raise an error instead of silently skipping the feature.
            for feature, importance in zip(feature_names, avg_importances):
                if np.isnan(importance):
                    raise ValueError(
                        f"{dataset_identifier} - SHAP importance of feature '{feature}' "
                        f"is NaN in every fold."
                    )
                importances_list.append({
                    'ballot': ballot,
                    'model': model_name,
                    'year': year,
                    'run': run_number,
                    'feature': feature,
                    'importance': importance
                })

            # 3. Dimensions and features used
            dimensions_list.append({
                'ballot': ballot,
                'model': model_name,
                'year': year,
                'run': run_number,
                'n_rows': len(X),
                'n_cols': len(feature_names)
            })

            feature_usage_list.append({
                'ballot': ballot,
                'model': model_name,
                'year': year,
                'run': run_number,
                'features_used': ','.join(feature_names),
                'con_features_used': ','.join([f for f in feature_names if f not in categorical_cols]),
                'nom_features_used': ','.join(categorical_cols)
            })
            # --- End of collecting results ---

        # --- End of year loop ---
    # --- End of (ballot, model) loop ---

    return pd.DataFrame(scores_list), pd.DataFrame(importances_list), pd.DataFrame(dimensions_list), pd.DataFrame(feature_usage_list)


# --- Repeat run_single_ml_iteration_weighted_top n_runs times and aggregate ---
def run_ml_multiple_times_weighted(
    data_for_analysis_dict, # input data: {(ballot, model): {year: DataFrame}}
    target_col='partyid_1',
    n_runs=3,
    use_weights=True,
    output_folder_name=None # OOF SHAP folder (passed on to run_single_ml_iteration_weighted_top)
):
    """
    Run run_single_ml_iteration_weighted_top n_runs times with random_seed = 0..n_runs-1
    and return the mean/SD of all runs by (ballot, year, model, metric).
    """
    all_scores_list = []
    all_importances_list = []
    all_dimensions_list = []
    all_feature_usage_list = []

    for i in range(n_runs):
        print(f"\n{'='*30}\n         RUN {i+1} / {n_runs}         \n{'='*30}")

        run_scores_df, run_importances_df, run_dimensions_df, run_feature_usage_df = run_single_ml_iteration_weighted_top(
            data_for_analysis_dict=data_for_analysis_dict,
            target_col=target_col,
            random_seed=i, # different seed per run, so CV splits and model initialization differ
            run_number=i+1,
            use_weights=use_weights,
            output_folder_name=output_folder_name
        )

        all_scores_list.append(run_scores_df)
        all_importances_list.append(run_importances_df)
        all_dimensions_list.append(run_dimensions_df)
        all_feature_usage_list.append(run_feature_usage_df)
        print(f"--- Run {i+1} completed. ---")

    # Combine the runs (ignoring empty results)
    scores_to_concat = [df for df in all_scores_list if isinstance(df, pd.DataFrame) and not df.empty]
    importances_to_concat = [df for df in all_importances_list if isinstance(df, pd.DataFrame) and not df.empty]
    dims_to_concat = [df for df in all_dimensions_list if isinstance(df, pd.DataFrame) and not df.empty]
    usage_to_concat = [df for df in all_feature_usage_list if isinstance(df, pd.DataFrame) and not df.empty]

    if not scores_to_concat:
        print("!! Warning: No scores generated. Returning empty results.")
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    all_scores_df = pd.concat(scores_to_concat, ignore_index=True)
    all_importances_df = pd.concat(importances_to_concat, ignore_index=True) if importances_to_concat else pd.DataFrame()
    all_dimensions_df = pd.concat(dims_to_concat, ignore_index=True) if dims_to_concat else pd.DataFrame()
    all_feature_usage_df = pd.concat(usage_to_concat, ignore_index=True) if usage_to_concat else pd.DataFrame()

    # Final summary: mean/SD over runs by (ballot, year, model, metric).
    final_summary = pd.DataFrame()
    if not all_scores_df.empty:
        # A failing aggregation (groupby/agg) means all_scores_df is malformed,
        # so the exception is propagated.
        final_summary = all_scores_df.groupby(['ballot', 'year', 'model', 'metric'])['avg_score'].agg(['mean', 'std']).reset_index()
        final_summary = final_summary.rename(columns={'mean': 'final_avg_score', 'std': 'final_std_score'})
        print(f"\n\n{'='*40}\n     FINAL AVERAGED RESULTS ({n_runs} RUNS)     \n{'='*40}")
        auc_summary = final_summary[final_summary['metric'] == 'roc_auc_ovr']
        if not auc_summary.empty:
            print(auc_summary.round(4).to_markdown(index=False))
        else:
            print("No ROC AUC scores.")

    return all_scores_df, all_importances_df, final_summary, all_dimensions_df, all_feature_usage_df


# --- Summary and saving ---
def compare_model_performance(scores_df, metric='roc_auc_ovr'):
    """Mean/SD of `metric` by (ballot, year, model), sorted."""
    print(f"\n{'='*30}\n   Model performance ({metric})   \n{'='*30}")

    if scores_df.empty or metric not in scores_df['metric'].values:
        print("!! Warning: Scores DataFrame empty/metric not found.")
        return pd.DataFrame(columns=['ballot', 'year', 'model', 'avg_score_mean', 'avg_score_std'])

    metric_df = scores_df[scores_df['metric'] == metric].copy()
    performance_summary = metric_df.groupby(['ballot', 'year', 'model'])['avg_score'].agg(['mean', 'std']).reset_index()
    performance_summary = performance_summary.rename(columns={'mean': 'avg_score_mean', 'std': 'avg_score_std'})
    performance_summary = performance_summary.sort_values(by=['ballot', 'year', 'avg_score_mean'], ascending=[True, True, False])
    print(performance_summary.round(4))
    return performance_summary




def analyze_feature_frequency(importance_df, top_n=5):
    """
    For each (ballot, model, run, year), take the top_n features by importance and count
    how often each feature is in the top N (frequency_top_n) and how often it is used at all (total_occurrences);
    frequency_normalized = frequency_top_n / total_occurrences.
    A high ratio means that whenever the feature is used in that (ballot, model) group,
    it almost always ranks near the top.
    """

    if importance_df.empty or 'importance' not in importance_df.columns:
        print("!! Warning: Importance DataFrame empty/missing importance column. Skipping frequency analysis.")
        return pd.DataFrame(), pd.DataFrame()

    # --- 0. Denominator base ---
    # With the {(ballot, model): {year: df}} structure, 'ballot' and 'year' columns always exist.
    denominator_base_df = importance_df.copy()
    top_n_group_keys = ['run', 'ballot', 'year', 'model']

    # --- 1. Top-N features (numerator base) ---
    # (rows used for 'frequency_top_n')
    top_features = importance_df.sort_values('importance', ascending=False).groupby(top_n_group_keys).head(top_n)

    numerator_base_df = top_features

    # --- 3. Overall frequency (not split by ballot/model) ---
    # Counts only; no normalization because groups are not split.
    overall_analysis = top_features.groupby('feature')['importance'].agg(
        frequency=('count'),
        importance_mean=('mean')
    ).reset_index().sort_values(by='frequency', ascending=False)

    # --- 4. By group: numerator ---
    grouped_analysis_numerator = numerator_base_df.groupby(['ballot', 'model', 'feature'])['importance'].agg(
        frequency_top_n=('count'), # (A) times in the top N
        importance_mean=('mean')
    ).reset_index()

    # --- 5. By group: denominator ---
    grouped_analysis_denominator = denominator_base_df.groupby(['ballot', 'model', 'feature'])['importance'].agg(
        total_occurrences=('count') # (B) times the variable was used
    ).reset_index()

    # --- 6. Merge (A) and (B), normalized frequency ---
    grouped_analysis = pd.merge(
        grouped_analysis_numerator,
        grouped_analysis_denominator,
        on=['ballot', 'model', 'feature'],
        how='left' # keyed on the variables in the top N
    )

    # (A) / (B)
    grouped_analysis['frequency_normalized'] = grouped_analysis['frequency_top_n'].astype(float) / grouped_analysis['total_occurrences']
    grouped_analysis['frequency_normalized'] = grouped_analysis['frequency_normalized'].fillna(0) # guard against NaN

    # --- 7. Sort by normalized frequency ---
    grouped_analysis = grouped_analysis.sort_values(
        by=['ballot', 'model', 'frequency_normalized'],
        ascending=[True, True, False]
    ).reset_index(drop=True)

    # --- 8. Print results by (ballot, model) ---
    print("\n--- Grouped Frequency Analysis (Normalized) ---")
    for name, group in grouped_analysis.groupby(['ballot', 'model']):
        ballot_name, model_name = name
        print(f"\n[Group: {ballot_name} - {model_name}]")
        # Columns to show
        print(group[[
            'feature',
            'frequency_normalized',
            'frequency_top_n',
            'total_occurrences',
            'importance_mean'
        ]].head(5).to_markdown(index=False, floatfmt=".4f"))

    return overall_analysis, grouped_analysis




def analyze_temporal_change(importance_df):
    print(f"\n{'='*40}\n   Feature importance by year (score + rank)   \n{'='*40}")

    if importance_df.empty or 'importance' not in importance_df.columns:
        print("!! Warning: Importance DataFrame empty. Skipping temporal analysis.")
        return {}

    df = importance_df.copy()
    df['importance'] = pd.to_numeric(df['importance'], errors='coerce')
    df = df.dropna(subset=['importance'])

    if df.empty:
        print("!! Warning: No valid numeric importance data for temporal analysis.")
        return {}

    avg_importance_df = df.groupby(['model', 'ballot', 'year', 'feature'])['importance'].mean().reset_index()
    avg_importance_df['rank'] = avg_importance_df.groupby(['model', 'ballot', 'year'])['importance'].rank(method='first', ascending=False).astype(int)

    results_by_model = {}
    for model_name in avg_importance_df['model'].unique():
        model_df = avg_importance_df[avg_importance_df['model'] == model_name]
        for ballot_type in sorted(model_df['ballot'].unique()):
            ballot_df = model_df[model_df['ballot'] == ballot_type]
            # A failing pivot (e.g. duplicated (feature, year) pairs) means importance_df
            # is malformed, so the exception is allowed to stop the analysis.
            pivot_table = ballot_df.pivot_table(index='feature', columns='year', values=['importance', 'rank'])
            pivot_table = pivot_table.swaplevel(0, 1, axis=1).sort_index(axis=1)
            results_by_model[f"Temporal_{model_name}_{ballot_type}"] = pivot_table
    return results_by_model




def analyze_cross_model_change(importance_df):
    print(f"\n{'='*40}\n   Feature importance by model and year   \n{'='*40}")

    if importance_df.empty or 'importance' not in importance_df.columns:
        print("!! Warning: Importance DataFrame empty. Skipping cross-model analysis.")
        return {}

    df = importance_df.copy()
    df['importance'] = pd.to_numeric(df['importance'], errors='coerce')
    df = df.dropna(subset=['importance'])

    if df.empty:
        print("!! Warning: No valid numeric importance data for cross-model analysis.")
        return {}

    avg_importance_df = df.groupby(['model', 'ballot', 'year', 'feature'])['importance'].mean().reset_index()
    avg_importance_df['rank'] = avg_importance_df.groupby(['model', 'ballot', 'year'])['importance'].rank(method='first', ascending=False).astype(int)

    results_by_ballot = {}
    for ballot_type in sorted(avg_importance_df['ballot'].unique()):
        ballot_df = avg_importance_df[avg_importance_df['ballot'] == ballot_type]
        # A failing pivot (e.g. duplicated (feature, year, model) triples) means importance_df
        # is malformed, so the exception is allowed to stop the analysis.
        pivot_table = ballot_df.pivot_table(index='feature', columns=['year', 'model'], values=['importance', 'rank'])
        pivot_table.columns = [f"{year}_{model}_{val}" for val, year, model in pivot_table.columns]
        pivot_table = pivot_table.sort_index(axis=1)
        results_by_ballot[f"CrossModel_{ballot_type}"] = pivot_table
    return results_by_ballot




def save_results_to_excel(results_dict, filename="ml_analysis_summary.xlsx"):
    """Save each item of results_dict as one sheet of an Excel file."""
    with pd.ExcelWriter(filename) as writer:
        for sheet_name, df in results_dict.items():
             if isinstance(df, pd.DataFrame):
                # Ensure sheet names are valid
                safe_sheet_name = str(sheet_name).replace('[','').replace(']','').replace('*','').replace(':','').replace('?','').replace('/','\\')
                safe_sheet_name = safe_sheet_name[:31] # Max length limit

                if isinstance(df.index, (pd.MultiIndex, pd.Index)) and df.index.name is not None:
                    df.to_excel(writer, sheet_name=safe_sheet_name, index=True)
                else:
                    df.to_excel(writer, sheet_name=safe_sheet_name, index=False)
    print(f"\n  All results saved to '{filename}'.")


# --- Full workflow: training -> performance/importance summaries -> Excel/Parquet ---


def run_analysis_for_target_weighted_selective(
    df_dict,          # input data: {(ballot, model): {year: DataFrame}}
    df_name,          # name used for the output files and the OOF SHAP folder
    target_col,       # target column
    n_runs            # number of repetitions of run_ml_multiple_times_weighted
):
    """
    Run the full workflow for target_col (training -> performance and feature importance
    -> changes by year/model) and save the results to one Excel file and three parquet files.
    df_name is used as the OOF SHAP folder name and the prefix of the output files.
    """
    print(f"\n{'='*60}")
    print(f"  Starting Workflow for Target Variable: '{target_col}' ({n_runs} runs)")
    print(f"  Data Source (df_name): {df_name}")
    print(f"{'='*60}")

    # Train/evaluate n_runs times; the OOF SHAP values of run 1 are saved in the df_name folder.
    all_scores, all_importances, final_summary, all_dimension, all_feature_usage = run_ml_multiple_times_weighted(
        data_for_analysis_dict=df_dict,
        target_col=target_col,
        n_runs=n_runs,
        use_weights=True, # GSS weights
        output_folder_name=os.path.join(OUT_DIR, 'shap', df_name)
    )

    performance_comparison_df = compare_model_performance(all_scores, metric='roc_auc_ovr')

    # all_importances has 'ballot', 'year', 'model' and 'run' columns,
    # This satisfies the 'ballot'/'year' columns required by analyze_feature_frequency.
    overall_freq_df, grouped_freq_df = analyze_feature_frequency(all_importances, top_n=20)
    temporal_results = analyze_temporal_change(all_importances)
    cross_model_results = analyze_cross_model_change(all_importances)

    print(f"\n--- [{target_col} - {df_name}] Collecting results for the Excel file... ---\n")

    # Mean sample size (n_rows) and number of features (n_cols) over runs by (ballot, year, model).
    unique_dimensions_df = pd.DataFrame()
    if not all_dimension.empty:
        # A failing aggregation (groupby/mean) means all_dimension is malformed,
        # so the exception is propagated.
        avg_dims = all_dimension.groupby(['ballot', 'year', 'model'])[['n_rows', 'n_cols']].mean().reset_index()
        unique_dimensions_df = avg_dims

    # Log of the features used in each model; identical across runs, so run 1 is used.
    unique_feature_usage_df = pd.DataFrame()
    if not all_feature_usage.empty:
        unique_feature_usage_df = all_feature_usage[all_feature_usage['run'] == 1].drop(columns=['run']).reset_index(drop=True)


    # Sheets of the Excel file: summary/performance/scores/feature frequency + changes by year/model
    excel_sheets = {
        "0_Final_Summary": final_summary,
        "1_Dataset_Dimensions": unique_dimensions_df,
        "2_Performance_Compare": performance_comparison_df,
        "3_Feature_Freq_Overall": overall_freq_df,
        "4_Feature_Freq_Grouped": grouped_freq_df,
        "5_Feature_Usage_Log": unique_feature_usage_df,
        **temporal_results,
        **cross_model_results
    }

    # df_name is used as the file prefix to identify the feature set.
    os.makedirs(OUT_DIR, exist_ok=True)
    excel_filename = os.path.join(OUT_DIR, f"{df_name}_{target_col}_analysis_results_weighted.xlsx")
    importance_filename = os.path.join(OUT_DIR, f"{df_name}_{target_col}_analysis_all_importance_weighted.parquet")
    scores_filename = os.path.join(OUT_DIR, f"{df_name}_{target_col}_analysis_all_scores_weighted.parquet")
    usage_log_filename = os.path.join(OUT_DIR, f"{df_name}_{target_col}_analysis_all_feature_usage.parquet")

    save_results_to_excel(excel_sheets, filename=excel_filename)

    # all_importances/all_scores/all_feature_usage have one row per (ballot x model x year x run),
    # so they are saved as parquet files rather than Excel.
    print(f"\n--- [{target_col} - {df_name}] Saving raw results to parquet... ---")
    if not all_importances.empty: all_importances.to_parquet(importance_filename)
    if not all_scores.empty: all_scores.to_parquet(scores_filename)
    if not all_feature_usage.empty: all_feature_usage.to_parquet(usage_log_filename)

    print(f"  [{target_col} - {df_name}] All files saved.")
    print(f"{'-'*60}\n")

## Settings and helpers for building the feature sets

In [ ]:
import pandas as pd, numpy as np, os

# ===== Common settings =====
N_ITERATIONS = 20
TARGET_COL = 'partyid_1'
ESSENTIAL_COLS = ['wtssnrps', 'wtssps', 'ballot']
MODEL_KEYS = [('a','XGBoost'),('b','XGBoost'),('c','XGBoost')]
FINAL_IMPORTANCE_FILE = "outputs/3_final/3_FINAL_M3_Top40_partyid_1_analysis_all_importance_weighted.parquet"

# Variable metadata (type = con/nom, attitude = 0/1, category)
# Only XGBoost is used, so the XGBoost variable list (XGBVARS = union of the a/b/c_XGB sheets) is used.

TYPE_SHEET = 'XGBVARS'
df_types = pd.read_excel("inputs/top_40_features_m3.xlsx", sheet_name=TYPE_SHEET)
var_type_map = df_types.set_index('var')['type'].to_dict()
att_set = set(df_types[df_types['attitude'] == 1]['var'].tolist())


def split_con_nom(features):
    """Split a feature list into {'con': [...], 'nom': [...]}; raise if a variable has no type."""
    missing = [f for f in features if f not in var_type_map]
    if missing:
        raise ValueError(f"Variables without type information in sheet '{TYPE_SHEET}': {missing}")
    return {
        'con': [f for f in features if var_type_map[f] == 'con'],
        'nom': [f for f in features if var_type_map[f] == 'nom'],
    }


def same_features_for_all(features):
    """feature_dict that assigns the same feature list to every (ballot, model)."""
    return {key: split_con_nom(features) for key in MODEL_KEYS}


def top_n_2020s(n, start_year=2020):
    """Top-n variables per (ballot, model) by mean final-model importance since start_year.
    Returns {(ballot, model): [feature, ...]} in descending order of importance."""
    df_imp = pd.read_parquet(FINAL_IMPORTANCE_FILE)
    avg = (df_imp[df_imp['year'] >= start_year]
           .groupby(['ballot','model','feature'])['importance'].mean().reset_index())
    top_df = (avg.sort_values(['ballot','model','importance'], ascending=[True,True,False])
                 .groupby(['ballot','model']).head(n))
    return {key: top_df[(top_df['ballot']==key[0]) & (top_df['model']==key[1])]['feature'].tolist()
            for key in MODEL_KEYS}


def build_custom_data(feature_dict):
    """Build {(ballot, model): {year: df}} from feature_dict {(ballot, model): {'con': [...], 'nom': [...]}}.
    nom -> category, con/target -> numeric. Year/ballot cells in which none of the selected variables is observed are skipped."""
    data_for_analysis_custom = {}
    for year, df_year in df_by_year.items():
        if year < 1988 or 'ballot' not in df_year.columns:
            continue
        df_y = df_year.copy()
        if df_y.columns.has_duplicates:
            df_y = df_y.loc[:, ~df_y.columns.duplicated()]
        for bc, bl in zip([1,2,3], ['a','b','c']):
            df_b = df_y[df_y['ballot'] == bc].copy()
            if df_b.empty: continue
            if TARGET_COL not in df_b.columns: continue
            valid = df_b[TARGET_COL].notna() & (df_b[TARGET_COL] != 0)
            for mn in ['XGBoost']:
                key = (bl, mn)
                if key not in feature_dict: continue
                con_f = feature_dict[key].get('con', [])
                nom_f = feature_dict[key].get('nom', [])
                # A split-ballot item can exist as a column in a year but be entirely missing in this ballot,
                # so only features observed for at least one respondent with a valid target are used.
                existing = [f for f in dict.fromkeys(con_f + nom_f)
                            if f in df_b.columns and df_b.loc[valid, f].notna().any()]
                if not existing:
                    continue  # expected: questionnaires differ by year/ballot
                keep = [TARGET_COL] + [c for c in ESSENTIAL_COLS if c in df_b.columns] + existing
                df_f = df_b[keep].copy()
                for col in df_f.columns:
                    if col in nom_f: df_f[col] = df_f[col].astype('category')
                    elif col in con_f or col == TARGET_COL: df_f[col] = pd.to_numeric(df_f[col], errors='coerce')
                if not df_f.empty:
                    data_for_analysis_custom.setdefault(key, {})[year] = df_f
    return data_for_analysis_custom


def run_featureset(analysis_name, feature_dict, n_runs=N_ITERATIONS):
    """Build the data from feature_dict and run run_analysis_for_target_weighted_selective.
    Output files: {analysis_name}_{TARGET_COL}_analysis_*"""
    if not feature_dict:
        print(f"[{analysis_name}] No features selected; not run.")
        return
    for key, fd in feature_dict.items():
        print(f"  {key}: con={fd['con']} nom={fd['nom']}")
    data_for_analysis_custom = build_custom_data(feature_dict)
    n_cells = sum(len(v) for v in data_for_analysis_custom.values())
    print(f"[{analysis_name}] Data ready ({n_cells} ballot-model-year cells)")
    run_analysis_for_target_weighted_selective(
        df_dict=data_for_analysis_custom,
        df_name=analysis_name,
        target_col=TARGET_COL,
        n_runs=n_runs
    )
    print(f"\n{'='*20}\n[{analysis_name}] done\n{'='*20}")


print("[OK] Settings and helpers defined")

## Top 3 variables of each ballot after 2020 (Figure 2-F)

In [ ]:
# ========== Top 3 variables of each ballot after 2020 ==========
top3 = top_n_2020s(3)
run_featureset("4_1_2020s_Parsimon_top3_only",
               {key: split_con_nom(feats) for key, feats in top3.items() if feats})

## Attitude variables only (Figure 2-D)

In [ ]:
# ========== Attitude variables only (attitude == 1) ==========
target_vars = df_types[df_types['attitude'] == 1]['var'].tolist()
print(f"  attitude==1 variables: {len(target_vars)}")
run_featureset("4_1_Final_Att_Only", same_features_for_all(target_vars))

## Without attitude variables (Figure 2-C)

In [ ]:
# ========== Without attitude variables (attitude == 0) ==========
target_vars = df_types[df_types['attitude'] == 0]['var'].tolist()
print(f"  attitude==0 variables: {len(target_vars)}")
run_featureset("4_1_Final_Att_Excluded", same_features_for_all(target_vars))

## Single variable: number of immigrants (Figure S4)

In [ ]:
# ========== Prediction with a single variable ==========
# Single variable used
SINGLE_FEATURE = 'letin1a'
run_featureset(f"4_1_Single_{SINGLE_FEATURE}", same_features_for_all([SINGLE_FEATURE]))

## Models by variable category (Figure 2-E)

In [ ]:
# ========== Models by variable category: Attitude / Demography / SES / Lifestyle + Social Capital ==========
# 1) attitude == 1 -> Attitude model
# 2) attitude == 0 -> one of three groups by the category column of XGBVARS (every non-attitude variable must belong to exactly one)
# 3) Each model is run separately to compare prediction performance (same variable set for every (ballot, model))
NON_ATT_GROUPS = {
    'Demography':      ['Demography'],
    'SES':             ['SES'],
    'LifestyleSoccap': ['Lifestyle', 'Social Capital'],
}

non_att = df_types[df_types['attitude'] == 0]
category_models = {'Attitude': df_types[df_types['attitude'] == 1]['var'].tolist()}
for name, cats in NON_ATT_GROUPS.items():
    category_models[name] = non_att[non_att['category'].isin(cats)]['var'].tolist()

# Stop if a non-attitude variable is not assigned to a group (typo, new category, etc.)
assigned = {v for name in NON_ATT_GROUPS for v in category_models[name]}
unassigned = non_att[~non_att['var'].isin(assigned)][['var', 'category']]
if not unassigned.empty:
    raise ValueError(f"Non-attitude variables not assigned to a group:\n{unassigned.to_string(index=False)}")

for name, feats in category_models.items():
    print(f"  {name} ({len(feats)}): {feats}")

# Models to run
# The Attitude model has the same variable set as 4_1_Final_Att_Only, so its results are used instead of running it again.
RUN_MODELS = ['Demography', 'SES', 'LifestyleSoccap']
for name in RUN_MODELS:
    run_featureset(f"4_1_Cat_{name}", same_features_for_all(category_models[name]))